# 2. Evaluating Reasoning Models

---

Packages used in this notebook:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_llm_from_scratch",
    "torch",
    "sympy",
    "tokenizers"  # Used by reasoning_llm_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_llm_from_scratch version: 0.1.0
torch version: 2.14.0
sympy version: 1.14.0
tokenizers version: 0.23.2


## 2.1. Loading the pre-trained model to generate text

We begin by loading the model that we want to evaluate.

In [2]:
from reasoning_llm_from_scratch.evaluate_reasoning_llm import load_model_and_tokenizer

In [3]:
from reasoning_llm_from_scratch.generate_text_pretrained_llm import get_device

WHICH_MODEL = "base"
device = get_device()

model, tokenizer = load_model_and_tokenizer(
    which_model=WHICH_MODEL,
    device=device,
    use_compile=True
)

Using Apple Silicon GPU (MPS)
✓ ../qwen3/qwen3-0.6B-base.pth already up-to-date


In [4]:
from reasoning_llm_from_scratch.generate_text_pretrained_llm import generate_text_basic_stream_cache
import torch

prompt = (
    r"If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?"
)

input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)

all_token_ids = []
for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=2048,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0)
    decoded_id = tokenizer.decode(token_id.tolist())
    print(
        decoded_id,
        end="",
        flush=True
    )
    all_token_ids.append(token_id)

all_tokens = tokenizer.decode(all_token_ids)

 To find the value of \( a^2 + b^2 \) given that \( a + b = 3 \) and \( ab = \frac{13}{6} \), we can use the following algebraic identity:

\[
a^2 + b^2 = (a + b)^2 - 2ab
\]

**Step 1:** Substitute the given values into the equation.

\[
a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
\]

**Step 2:** Calculate \( (3)^2 \).

\[
(3)^2 = 9
\]

**Step 3:** Calculate \( 2 \times \frac{13}{6} \).

\[
2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3}
\]

**Step 4:** Subtract the second result from the first.

\[
a^2 + b^2 = 9 - \frac{13}{3}
\]

**Step 5:** Convert 9 to a fraction with a denominator of 3 to perform the subtraction.

\[
9 = \frac{27}{3}
\]

\[
a^2 + b^2 = \frac{27}{3} - \frac{13}{3} = \frac{14}{3}
\]

**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]

In [5]:
import re
from IPython.display import Markdown, display

def to_vscode_latex(text: str) -> str:
    text = re.sub(r'\\\[(.*?)\\\]', r'$$\1$$', text, flags=re.DOTALL)
    text = re.sub(r'\\\((.*?)\\\)', r'$\1$', text, flags=re.DOTALL)
    return text

display(Markdown(to_vscode_latex(all_tokens)))

 To find the value of $ a^2 + b^2 $ given that $ a + b = 3 $ and $ ab = \frac{13}{6} $, we can use the following algebraic identity:

$$
a^2 + b^2 = (a + b)^2 - 2ab
$$

**Step 1:** Substitute the given values into the equation.

$$
a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
$$

**Step 2:** Calculate $ (3)^2 $.

$$
(3)^2 = 9
$$

**Step 3:** Calculate $ 2 \times \frac{13}{6} $.

$$
2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3}
$$

**Step 4:** Subtract the second result from the first.

$$
a^2 + b^2 = 9 - \frac{13}{3}
$$

**Step 5:** Convert 9 to a fraction with a denominator of 3 to perform the subtraction.

$$
9 = \frac{27}{3}
$$

$$
a^2 + b^2 = \frac{27}{3} - \frac{13}{3} = \frac{14}{3}
$$

**Final Answer:**

$$
\boxed{\dfrac{14}{3}}
$$

## 2.2. Implementing a wrapper for easier text generation

For convenience, we will create a wrapper function for the text generation function so that we only have to pass in the model, tokenizer, and prompt, along with some additional settings.

In [6]:
from reasoning_llm_from_scratch.evaluate_reasoning_llm import generate_text_stream_concat

In [7]:
generated_text = generate_text_stream_concat(
    model, 
    tokenizer, 
    prompt, 
    device,
    max_new_tokens=2048,
    verbose=True
)   

 To find the value of \( a^2 + b^2 \) given that \( a + b = 3 \) and \( ab = \frac{13}{6} \), we can use the following algebraic identity:

\[
a^2 + b^2 = (a + b)^2 - 2ab
\]

**Step 1:** Substitute the given values into the equation.

\[
a^2 + b^2 = (3)^2 - 2 \left( \frac{13}{6} \right)
\]

**Step 2:** Calculate \( (3)^2 \).

\[
(3)^2 = 9
\]

**Step 3:** Calculate \( 2 \times \frac{13}{6} \).

\[
2 \times \frac{13}{6} = \frac{26}{6} = \frac{13}{3}
\]

**Step 4:** Subtract the second result from the first.

\[
a^2 + b^2 = 9 - \frac{13}{3}
\]

**Step 5:** Convert 9 to a fraction with a denominator of 3 to perform the subtraction.

\[
9 = \frac{27}{3}
\]

\[
a^2 + b^2 = \frac{27}{3} - \frac{13}{3} = \frac{14}{3}
\]

**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]

## 2.3. Extracting the final answer box

In [8]:
from reasoning_llm_from_scratch.evaluate_reasoning_llm import get_last_boxed

In [9]:
print(get_last_boxed(generated_text))

\dfrac{14}{3}


We can make answer extraction a bit more robust with the `extract_final_candidate` function, which accounts for cases where a final answer box is missing or incomplete.

In [10]:
from reasoning_llm_from_scratch.evaluate_reasoning_llm import extract_final_candidate

In [11]:
print(get_last_boxed(generated_text))
print(extract_final_candidate(r"\boxed{ 14/3. }"))
print(extract_final_candidate("abc < > 14/3 abc"))
print(extract_final_candidate("Text without numbers"))

\dfrac{14}{3}
14/3.
14/3
Text without numbers


## 2.4. Normalizing the extracted answer

In [12]:
from reasoning_llm_from_scratch.evaluate_reasoning_llm import normalize_text

In [13]:
print(normalize_text(extract_final_candidate(generated_text)))
print(normalize_text(r"$\dfrac{14}{3.}$"))
print(normalize_text(r"\text{\[\frac{14}{3}\]}"))
print(normalize_text("4/3"))


(14)/(3)
(14)/(3.)
(14)/(3)
4/3


## 2.5. Verifying mathematical equivalence

Now we implement the basic functionality to check if the extracted answer (generated by the model) is equivalent to the correct answer (ground truth) provided in the dataset.

In [14]:
from reasoning_llm_from_scratch.evaluate_reasoning_llm import sympy_parser

In [15]:
print(sympy_parser(normalize_text(
    extract_final_candidate(generated_text)
)))

14/3


In [16]:
from reasoning_llm_from_scratch.evaluate_reasoning_llm import equality_check

In [17]:
print(equality_check(
    normalize_text("13/4."),
    normalize_text(r"(13)/(4)")
))

print(equality_check(
    normalize_text("0.5"),
    normalize_text(r"(1)/(2)")
))

print(equality_check(
    normalize_text("14/3"),
    normalize_text("15/3")
))

print(equality_check(
    normalize_text("(14/3, 2/3)"),
    normalize_text("(14/3, 4/6)")
))

True
True
False
False


We will build on the mathematical equality-checking function to implement a robust grading function that can also handle tuple-like expressions, such as correctly comparing "(14/3, 2/3)" and "(14/3, 4/6)".

In [18]:
from reasoning_llm_from_scratch.evaluate_reasoning_llm import split_into_parts

In [19]:
split_into_parts(normalize_text(r"(14/3, 2/3)"))

['14/3', '2/3']

## 2.6. Grading answers

Now, we can implement the `grade_answer` function, which splits tuple-like expressions (if present) into subparts and then uses the `equality_check` function from to compare a generated answer to a reference (ground truth) answer.

In [20]:
from reasoning_llm_from_scratch.evaluate_reasoning_llm import grade_answer

In [21]:
grade_answer("14/3", r"\frac{14}{3}")
grade_answer(r"(14/3, 2/3)", "(14/3, 4/6)")

True

To check the grade_answer function more comprehensively, the following listing con-
tains more diverse test cases.

In [22]:
# Define test cases: (name, prediction, ground truth, expected result)
tests = [
        ("check_1", "3/4", r"\frac{3}{4}", True),
        ("check_2", "(3)/(4)", r"3/4", True),
        ("check_3", r"\frac{\sqrt{8}}{2}", "sqrt(2)", True),
        ("check_4", r"\( \frac{1}{2} + \frac{1}{6} \)", "2/3", True),
        ("check_5", "(1, 2)", r"(1,2)", True),
        ("check_6", "(2, 1)", "(1, 2)", False),
        ("check_7", "(1, 2, 3)", "(1, 2)", False),
        ("check_8", "0.5", "1/2", True),
        ("check_9", "0.3333333333", "1/3", False),
        ("check_10", "1,234/2", "617", True),
        ("check_11", r"\text{2/3}", "2/3", True),
        ("check_12", "50%", "1/2", False),
        ("check_13", r"2\cdot 3/4", "3/2", True),
        ("check_14", r"90^\circ", "90", True),
        ("check_15", r"\left(\frac{3}{4}\right)", "3/4", True),
        ("check_16", r"2²", "2**2", True),
    ]

from reasoning_llm_from_scratch.evaluate_reasoning_llm import run_demos_table

In [23]:
run_demos_table(tests)

Test     | Expect | Got   | Status
check_1  | True   | True  | PASS  
check_2  | True   | True  | PASS  
check_3  | True   | True  | PASS  
check_4  | True   | True  | PASS  
check_5  | True   | True  | PASS  
check_6  | False  | False | PASS  
check_7  | False  | False | PASS  
check_8  | True   | True  | PASS  
check_9  | False  | False | PASS  
check_10 | True   | True  | PASS  
check_11 | True   | True  | PASS  
check_12 | False  | False | PASS  
check_13 | True   | True  | PASS  
check_14 | True   | True  | PASS  
check_15 | True   | True  | PASS  
check_16 | True   | True  | PASS  

Passed 16/16


## 2.7. Loading the evaluation dataset

We’re now ready to evaluate the LLM on the MATH-500 benchmark dataset (https://huggingface.co/datasets/HuggingFaceH4/MATH-500), a widely used benchmark for reasoning models. It is a curated collection of 500 problems sampled from the original MATH dataset.

In [24]:
from reasoning_llm_from_scratch.evaluate_reasoning_llm import load_math500_test

math_data = load_math500_test()
print("Number of entries:", len(math_data))

Number of entries: 500


In [25]:
from pprint import pprint
pprint(math_data[0])

{'answer': '\\left( 3, \\frac{\\pi}{2} \\right)',
 'level': 2,
 'problem': 'Convert the point $(0,3)$ in rectangular coordinates to polar '
            'coordinates.  Enter your answer in the form $(r,\\theta),$ where '
            '$r > 0$ and $0 \\le \\theta < 2 \\pi.$',
 'solution': 'We have that $r = \\sqrt{0^2 + 3^2} = 3.$  Also, if we draw the '
             'line connecting the origin and $(0,3),$ this line makes an angle '
             'of $\\frac{\\pi}{2}$ with the positive $x$-axis.\n'
             '\n'
             '[asy]\n'
             'unitsize(0.8 cm);\n'
             '\n'
             'draw((-0.5,0)--(3.5,0));\n'
             'draw((0,-0.5)--(0,3.5));\n'
             'draw(arc((0,0),3,0,90),red,Arrow(6));\n'
             '\n'
             'dot((0,3), red);\n'
             'label("$(0,3)$", (0,3), W);\n'
             'dot((3,0), red);\n'
             '[/asy]\n'
             '\n'
             'Therefore, the polar coordinates are $\\boxed{\\left( 3, '
             '\\frac

## 2.8. Evaluating the model

In [26]:
from reasoning_llm_from_scratch.evaluate_reasoning_llm import render_prompt

In [27]:
prompt = (  # Same prompt we used at the beginning
    r"If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?"
)
prompt_fmt = render_prompt(prompt)
print(prompt_fmt)

You are a helpful math assistant.
Answer the question and write the final result on a new line as:
\boxed{ANSWER}

Question:
If $a+b=3$ and $ab=\tfrac{13}{6}$, what is the value of $a^2+b^2$?

Answer:


In [28]:
generated_text = generate_text_stream_concat(
    model, tokenizer, prompt_fmt, device,
    max_new_tokens=2048,
    verbose=True
)

 \boxed{10}

While brevity can speed up generation by reducing the number of tokens, the response is incorrect. Previously, by contrast, without a prompt template, the model
produced a longer response, which led to the correct answer, 14/3.

Whether the prompt template is well suited to a given model and task needs to be
determined on a larger set of examples, such as the MATH-500 dataset, before we can
draw any conclusions.

In [29]:
from reasoning_llm_from_scratch.evaluate_reasoning_llm import mini_eval_demo

In [30]:
mini_eval_demo(model, tokenizer, device)

Device: mps
Prediction: 1/3
Ground truth: 2/3
Correct: False


## 2.9. End-to-end model evaluation pipeline for MATH-500 dataset

In [31]:
from reasoning_llm_from_scratch.evaluate_reasoning_llm import evaluate_math500_stream

In [32]:
print("Model:", WHICH_MODEL)
print("Device:", device)
num_correct, num_examples, acc = evaluate_math500_stream(
    model, tokenizer, device, 
    math_data=math_data[:10],
    max_new_tokens=2048,
    verbose=False
)

Model: base
Device: mps
MATH-500: 10/10 | ETA: 00s        
Accuracy: 30.0% (3/10)
Total time: 0.2 min
Logs written to: ../math500-mps.jsonl
